# Ficha automática de mercado y entorno – Venture One SAS

**Reto 13 · EstudIA · Análisis de datos**

Este cuaderno genera, para cada inmueble de prueba, la **ficha JSON** que pide Venture One
(`docs/plantilla_ficha.md`) y la **valida** contra el cálculo manual (`validacion/esperado.json`).
Además deja listo el modelo para el tablero de **Power BI**.

| Parte | Qué hace |
|---|---|
| A | Carga los parámetros oficiales y los 5 archivos |
| B | Limpia los anuncios **con las reglas de `config/parametros.yaml`** |
| C | Mercado: comparables, estadísticas y confianza |
| D | Entorno: estación más cercana y equipamientos en el radio |
| E | Norma urbanística (POT) |
| F | Arma y guarda la ficha JSON de cada inmueble |
| G | **Valida** contra el cálculo manual |
| H | Exporta el modelo para Power BI |

**Regla de oro del reto:** *ningún número va fijo en el código*. Radio, estados, tolerancias y regla de
confianza se leen de `config/parametros.yaml`. Si la empresa cambia una regla, se edita ese archivo
y se vuelve a correr todo — eso es lo que hace la solución **sostenible**.

**Cómo usarlo:** ejecuta las celdas de arriba hacia abajo (`Shift + Enter`) o usa **Run All**.

---
# PARTE A. Parámetros y datos

## Paso 1. Librerías y rutas

Estructura de carpetas que espera el cuaderno (la del repositorio):

```
config/parametros.yaml            ← reglas oficiales
datos/originales/*.csv            ← 3 CSV
datos/originales/geo/*.geojson    ← transporte y equipamientos (y el POT cuando llegue)
validacion/esperado.json, validar.py
```

In [ ]:
import csv
import json
import math
import statistics
import unicodedata
from collections import Counter
from datetime import date
from pathlib import Path

import pandas as pd
import yaml   # para leer el archivo de parámetros (.yaml)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

RUTA_PARAMETROS = Path("config") / "parametros.yaml"
CARPETA_DATOS = Path("datos") / "originales"
CARPETA_GEO = CARPETA_DATOS / "geo"
CARPETA_FICHAS = Path("salidas") / "fichas"
CARPETA_POWERBI = Path("datos") / "limpios" / "powerbi"

for carpeta in (CARPETA_FICHAS, CARPETA_POWERBI):
    carpeta.mkdir(parents=True, exist_ok=True)

for ruta in (RUTA_PARAMETROS, CARPETA_DATOS / "anuncios_muestra.csv", CARPETA_GEO / "transporte.geojson"):
    print(("✅ " if ruta.exists() else "❌ FALTA ") + str(ruta))

## Paso 2. Leer los parámetros

Un archivo `.yaml` es texto con formato `clave: valor`. `yaml.safe_load` lo convierte en un diccionario de Python.
Mira la salida: ahí están el radio (800 m), los 3 estados canónicos y la regla de confianza.

In [ ]:
PARAMS = yaml.safe_load(RUTA_PARAMETROS.read_text(encoding="utf-8"))

print("Versión de parámetros:", PARAMS["version"])
print("Radio de búsqueda:", PARAMS["entorno"]["radio_busqueda_m"], "m")
print("Tolerancia de área:", PARAMS["comparables"]["tolerancia_area"])
print("Estados canónicos:", list(PARAMS["estados"]))
print("Confianza:", PARAMS["confianza"])

## Paso 3. Cargar los CSV

El separador puede ser coma o punto y coma (según quién guardó el archivo), así que lo **detectamos** mirando el encabezado.
Leemos todo como texto para que nada se transforme sin que lo decidamos.

In [ ]:
def leer_csv(nombre):
    ruta = CARPETA_DATOS / nombre
    with open(ruta, encoding="utf-8-sig") as f:
        encabezado = f.readline()
    sep = ";" if encabezado.count(";") > encabezado.count(",") else ","
    df = pd.read_csv(ruta, sep=sep, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    print(f"{nombre}: {len(df)} filas x {df.shape[1]} columnas (separador '{sep}')")
    return df.to_dict("records")   # lista de diccionarios: una fila = un diccionario

ANUNCIOS_CRUDOS = leer_csv("anuncios_muestra.csv")
INMUEBLES = leer_csv("inmuebles_prueba.csv")
INMUEBLES_FUENTE = leer_csv("inmuebles_prueba_fuente.csv")

ANUNCIOS_CRUDOS[0]

## Paso 4. Cargar las capas geográficas

En GeoJSON las coordenadas vienen como **[longitud, latitud]** (al revés de lo habitual).
La capa del POT todavía no fue entregada: el código la usa automáticamente cuando aparezca el archivo.

In [ ]:
def leer_geojson(ruta):
    return json.loads(Path(ruta).read_text(encoding="utf-8"))

TRANSPORTE = leer_geojson(CARPETA_GEO / "transporte.geojson")
EQUIPAMIENTOS = leer_geojson(CARPETA_GEO / "equipamientos.geojson")

RUTA_POT = CARPETA_GEO / Path(PARAMS["norma"]["archivo_pot"]).name
POT = leer_geojson(RUTA_POT) if RUTA_POT.exists() else None

print("Estaciones:", len(TRANSPORTE["features"]), "| Equipamientos:", len(EQUIPAMIENTOS["features"]))
print("Fecha de la foto de OSM:", TRANSPORTE["metadata"]["fecha_osm"])
print("Capa POT:", "cargada" if POT else f"pendiente (falta {RUTA_POT})")

---
# PARTE B. Limpieza con las reglas oficiales

Llevamos un **registro de calidad**: cada problema queda anotado (archivo, fila, qué pasó, qué se hizo).
Es la respuesta a uno de los dolores de la empresa: *"no deja registro de los datos faltantes"*.

In [ ]:
REGISTRO = []

def anotar(archivo, referencia, problema, detalle="", accion=""):
    REGISTRO.append({"archivo": archivo, "referencia": referencia, "problema": problema,
                     "detalle": str(detalle), "accion": accion})

## Paso 5. Normalizar el estado

Regla de `parametros.yaml`: minúsculas, sin tildes, sin `?`, sin espacios sobrantes; luego se busca en la lista de cada estado.

| Texto original | Normalizado | Estado canónico |
|---|---|---|
| `REMODELADA ?` | `remodelada` | `remodelado` |
| `OK`, `Vivir`, `HABITABLE POR REMODELAR` | … | `habitable` |
| `Remodelar`, `Reformar`, `SIN REMODELAR (OBRA NEGRA)` | … | `por_remodelar` |
| *(vacío)* o desconocido | | `sin_clasificar` → no sirve como comparable |

In [ ]:
def texto_normalizado(texto):
    sin_tildes = unicodedata.normalize("NFKD", str(texto or "")).encode("ascii", "ignore").decode()
    return " ".join(sin_tildes.replace("?", " ").lower().split())

def normalizar_estado(texto):
    limpio = texto_normalizado(texto)
    for canonico, variantes in PARAMS["estados"].items():
        if limpio in [texto_normalizado(v) for v in variantes]:
            return canonico
    return "sin_clasificar"

for ejemplo in ["REMODELADA ?", "OK", "Vivir", "SIN REMODELAR (OBRA NEGRA)", "Reformar", ""]:
    print(f"{ejemplo!r:30} → {normalizar_estado(ejemplo)}")

## Paso 6. Preparar cada anuncio

Para cada fila: convertimos área y precio a número (si no es un número positivo → `None`, que en Python significa "sin dato"),
calculamos el **precio por m²** y el estado canónico. También quitamos caracteres extraños del `id_anuncio`
(hay uno que termina en `#`).

In [ ]:
import re

def a_numero(valor):
    try:
        n = float(valor)
    except (TypeError, ValueError):
        return None
    return n if n > 0 else None

def preparar(fila):
    area, precio = a_numero(fila["area_m2"]), a_numero(fila["precio"])
    id_limpio = re.sub(r"[^A-Za-z0-9_:\-]", "", fila["id_anuncio"])
    if id_limpio != fila["id_anuncio"]:
        anotar("anuncios_muestra.csv", fila["origen"], "id con caracteres no válidos",
               fila["id_anuncio"], f"corregido a {id_limpio}")
    return {**fila,
            "id_anuncio": id_limpio,
            "area_m2": area,
            "precio": precio,
            "precio_m2": precio / area if area and precio else None,
            "estado_canonico": normalizar_estado(fila["estado"])}

CRUDOS = [preparar(f) for f in ANUNCIOS_CRUDOS]
pd.DataFrame(CRUDOS)[["origen", "estado", "estado_canonico", "area_m2", "precio", "precio_m2"]].head(8)

## Paso 7. Eliminar duplicados

Regla oficial (`deduplicacion` en los parámetros). Recorremos el CSV **en orden** y una fila es duplicada si:
1. su `id_anuncio` (no vacío) ya apareció, **o**
2. su combinación (`zona`, `area_m2`, `precio`) ya apareció → detecta el mismo inmueble publicado en dos portales o en Facebook.

Se conserva **la primera** aparición. Cada anuncio único guarda la lista de filas que "absorbió"
(`origenes_agrupados`): así no perdemos la trazabilidad y podemos saber si un comparable es el propio inmueble.

In [ ]:
def deduplicar(anuncios):
    columnas = PARAMS["deduplicacion"]["claves_sin_id"]
    unicos, por_id, por_clave = [], {}, {}
    for a in anuncios:
        valores = tuple(a.get(c) for c in columnas)
        clave = None if any(v in (None, "") for v in valores) else valores
        destino = por_id.get(a["id_anuncio"]) if a["id_anuncio"] else None
        if destino is None and clave is not None:
            destino = por_clave.get(clave)
        if destino is None:                       # primera vez que lo vemos: es un anuncio único
            destino = {**a, "origenes_agrupados": [], "ids_agrupados": []}
            unicos.append(destino)
        else:                                     # ya existía: es duplicado
            anotar("anuncios_muestra.csv", a["origen"], "duplicado eliminado",
                   f"igual a {destino['origen']}", "se conserva la primera aparición")
        destino["origenes_agrupados"].append(a["origen"])
        if a["id_anuncio"] and a["id_anuncio"] not in destino["ids_agrupados"]:
            destino["ids_agrupados"].append(a["id_anuncio"])
        if a["id_anuncio"]:
            por_id.setdefault(a["id_anuncio"], destino)
        if clave is not None:
            por_clave.setdefault(clave, destino)
    return unicos

ANUNCIOS = deduplicar(CRUDOS)
print(f"Filas: {len(CRUDOS)} → anuncios únicos: {len(ANUNCIOS)} ({len(CRUDOS) - len(ANUNCIOS)} duplicados)")

# Casos para discutir: un mismo grupo con ids de portales DISTINTOS (¿mismo inmueble o unidades iguales?)
for a in ANUNCIOS:
    if len(a["ids_agrupados"]) > 1:
        anotar("anuncios_muestra.csv", a["origen"], "duplicado ambiguo: ids distintos con misma zona, área y precio",
               ", ".join(a["ids_agrupados"]), "revisar manualmente")
        print("⚠ Ambiguo:", a["ids_agrupados"])

## Paso 8. Marcar los anuncios inválidos

Un anuncio no sirve como comparable si (en este orden): no tiene área, no tiene precio,
su precio/m² está fuera de **[1.000.000 ; 20.000.000]**, su estado quedó `sin_clasificar` o el tipo de precio no es aceptado.
No se borran: quedan marcados con el **motivo**.

In [ ]:
def motivo_invalido(a):
    if a["area_m2"] is None:
        return "sin_area"
    if a["precio"] is None:
        return "sin_precio"
    minimo, maximo = PARAMS["comparables"]["rango_precio_m2_valido"]
    if not minimo <= a["precio_m2"] <= maximo:
        return "precio_m2_fuera_de_rango"
    if a["estado_canonico"] == "sin_clasificar":
        return "estado_sin_clasificar"
    if a["tipo_precio"] not in PARAMS["comparables"]["tipos_precio"]:
        return "tipo_precio_no_aceptado"
    return None

for a in ANUNCIOS:
    a["motivo_invalido"] = motivo_invalido(a)
    if a["motivo_invalido"]:
        anotar("anuncios_muestra.csv", a["origen"], "anuncio inválido", a["motivo_invalido"], "no se usa como comparable")

# Otras observaciones de calidad (no cambian el cálculo, pero se registran)
for a in ANUNCIOS:
    if a["zona"] == "ditaires_itagui" and a["municipio"] != "itagui":
        anotar("anuncios_muestra.csv", a["origen"], "municipio no coincide con la zona", a["municipio"], "revisar")
anotar("anuncios_muestra.csv", "todas", "columna fecha vacía", "", "no se puede medir antigüedad ni liquidez")
anotar("anuncios_muestra.csv", "todas", "sin precios de cierre", "todas las filas son 'pedido'", "la referencia es de oferta, no de venta")

pd.crosstab(pd.Series([a["zona"] for a in ANUNCIOS], name="zona"),
            pd.Series([a["motivo_invalido"] or "válido" for a in ANUNCIOS], name="resultado"))

---
# PARTE C. Mercado

## Paso 9. Seleccionar comparables

Un anuncio es comparable de un inmueble si:
- es **válido**,
- está en la **misma zona** y el **mismo estado canónico**,
- su área está dentro de **±30 %** del área del inmueble (`tolerancia_area`),
- **no es el propio inmueble** (ni ninguna fila de su grupo de duplicados).

In [ ]:
def es_el_mismo_inmueble(anuncio, inmueble):
    return (inmueble["origen"] in anuncio["origenes_agrupados"]
            or (inmueble["id_anuncio"] and inmueble["id_anuncio"] in anuncio["ids_agrupados"]))

def seleccionar_comparables(inmueble):
    tol = PARAMS["comparables"]["tolerancia_area"]
    return [a for a in ANUNCIOS
            if a["motivo_invalido"] is None
            and a["zona"] == inmueble["zona"]
            and a["estado_canonico"] == inmueble["estado_canonico"]
            and abs(a["area_m2"] - inmueble["area_m2"]) <= tol * inmueble["area_m2"]
            and not es_el_mismo_inmueble(a, inmueble)]

# Preparamos los inmuebles: números y estado canónico
for inm in INMUEBLES:
    inm["area_m2"] = float(inm["area_m2"])
    inm["lat"], inm["lon"] = float(inm["lat"]), float(inm["lon"])
    inm["estado_canonico"] = normalizar_estado(inm["estado"])

ejemplo = next(i for i in INMUEBLES if i["id_inmueble"] == "INM-04")
print("INM-04 →", len(seleccionar_comparables(ejemplo)), "comparables")

## Paso 10. Estadísticas y regla de confianza

- **Mediana**: el valor del medio (no se deja "jalar" por valores extremos).
- **Desviación estándar muestral** (divide por n − 1) y **coeficiente de variación** (CV = desviación ÷ promedio):
  mide qué tan dispersos están los precios. CV bajo = comparables parecidos = más confianza.
- **Confianza**: *alta* si n ≥ 5 y CV ≤ 15 %; *media* si n ≥ 3 y CV ≤ 25 %; si no, *baja*.

Precios redondeados a peso entero y CV a 4 decimales, como pide la plantilla.

In [ ]:
def estadisticas(valores):
    n = len(valores)
    if n == 0:
        return {"n": 0, "mediana": None, "promedio": None, "minimo": None, "maximo": None,
                "desviacion": None, "coef_variacion": None}
    promedio = statistics.mean(valores)
    desviacion = statistics.stdev(valores) if n >= 2 else None
    return {"n": n,
            "mediana": round(statistics.median(valores)),
            "promedio": round(promedio),
            "minimo": round(min(valores)),
            "maximo": round(max(valores)),
            "desviacion": round(desviacion) if desviacion is not None else None,
            "coef_variacion": round(desviacion / promedio, 4) if desviacion is not None else None}

def nivel_confianza(n, cv):
    for nivel in ("alta", "media"):
        regla = PARAMS["confianza"][nivel]
        if n >= regla["min_comparables"] and cv is not None and cv <= regla["max_coef_variacion"]:
            return nivel, (f"{n} comparables (mínimo {regla['min_comparables']}) y coeficiente de variación "
                           f"{cv:.2%} (máximo {regla['max_coef_variacion']:.0%})")
    m = PARAMS["confianza"]["media"]
    cv_txt = "sin dato" if cv is None else f"{cv:.2%}"
    return "baja", (f"{n} comparables y coeficiente de variación {cv_txt}: no cumple el nivel media "
                    f"(mínimo {m['min_comparables']} comparables, máximo {m['max_coef_variacion']:.0%})")

def bloque_mercado(inmueble):
    comparables = sorted(seleccionar_comparables(inmueble), key=lambda a: a["precio_m2"])
    est = estadisticas([a["precio_m2"] for a in comparables])
    nivel, motivo = nivel_confianza(est["n"], est["coef_variacion"])
    resumen = {}
    for estado in PARAMS["estados"]:              # contexto: mediana por estado en toda la zona (sin filtro de área)
        valores = [a["precio_m2"] for a in ANUNCIOS
                   if a["motivo_invalido"] is None and a["zona"] == inmueble["zona"]
                   and a["estado_canonico"] == estado and not es_el_mismo_inmueble(a, inmueble)]
        e = estadisticas(valores)
        resumen[estado] = {"n": e["n"], "mediana": e["mediana"]}
    return {
        "estado_referencia": inmueble["estado_canonico"],
        "precio_m2": est,
        "confianza": {"nivel": nivel, "motivo": motivo},
        "comparables": [{"origen": a["origen"], "id_anuncio": a["id_anuncio"], "fuente": a["fuente"],
                         "area_m2": a["area_m2"], "precio": round(a["precio"]), "precio_m2": round(a["precio_m2"])}
                        for a in comparables],
        "precio_m2_por_estado_en_zona": resumen,
    }

bloque_mercado(ejemplo)["precio_m2"]

## Paso 11. Calidad de datos por zona

Cada ficha declara cuántos anuncios había en su zona, cuántos eran duplicados y cuántos inválidos (por motivo).

In [ ]:
def calidad_zona(zona):
    crudos = [a for a in CRUDOS if a["zona"] == zona]
    unicos = [a for a in ANUNCIOS if a["zona"] == zona]
    invalidos = Counter(a["motivo_invalido"] for a in unicos if a["motivo_invalido"])
    return {"anuncios_en_zona": len(crudos),
            "duplicados_eliminados": len(crudos) - len(unicos),
            "anuncios_unicos": len(unicos),
            "anuncios_invalidos": dict(sorted(invalidos.items())),
            "anuncios_validos": len(unicos) - sum(invalidos.values())}

{z: calidad_zona(z) for z in sorted({a["zona"] for a in CRUDOS})}

---
# PARTE D. Entorno

## Paso 12. Distancia haversine

Calcula la distancia en metros sobre la superficie de la Tierra (radio 6.371.008,8 m, el del cálculo manual).

In [ ]:
RADIO_TIERRA_M = 6_371_008.8

def haversine_m(lat1, lon1, lat2, lon2):
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * RADIO_TIERRA_M * math.asin(math.sqrt(a))

def distancia_a(lat, lon, feature):
    flon, flat = feature["geometry"]["coordinates"]     # ¡primero longitud!
    return haversine_m(lat, lon, flat, flon)

print(round(haversine_m(6.0, -75.0, 7.0, -75.0)), "m en un grado de latitud (≈ 111.195)")

## Paso 13. Bloque de entorno

- **Estación más cercana** entre los sistemas que cuentan (`metro`, `tranvia`, `metrocable`; metroplús no, porque OSM solo tiene 1 estación).
- **Estaciones y equipamientos a 800 m o menos**, por categoría.
- `aproximado = true` si la coordenada es del barrio y no de la dirección.

> Nota: la capa de equipamientos se usa **tal como la entregó la empresa** (así lo hace el cálculo manual).
> Los puntos sin nombre o repetidos en OSM se reportan en el registro de calidad, pero no se eliminan del conteo.

In [ ]:
def bloque_entorno(inmueble):
    p = PARAMS["entorno"]
    lat, lon, radio = inmueble["lat"], inmueble["lon"], p["radio_busqueda_m"]
    estaciones = sorted(((distancia_a(lat, lon, f), f) for f in TRANSPORTE["features"]
                         if f["properties"]["sistema"] in p["sistemas_transporte"]), key=lambda par: par[0])
    cercana = None
    if estaciones:
        d, f = estaciones[0]
        cercana = {"nombre": f["properties"]["nombre"], "sistema": f["properties"]["sistema"], "distancia_m": round(d)}
    conteo = {cat: 0 for cat in p["categorias_equipamientos"]}
    for f in EQUIPAMIENTOS["features"]:
        if f["properties"]["categoria"] in conteo and distancia_a(lat, lon, f) <= radio:
            conteo[f["properties"]["categoria"]] += 1
    return {"radio_busqueda_m": radio,
            "estacion_mas_cercana": cercana,
            "estaciones_en_radio": sum(1 for d, _ in estaciones if d <= radio),
            "equipamientos_en_radio": conteo,
            "aproximado": inmueble["precision_coordenadas"] != "cruce"}

# Observaciones de calidad de las capas
sin_nombre = sum(1 for f in EQUIPAMIENTOS["features"] if not f["properties"]["nombre"].strip())
anotar("equipamientos.geojson", "capa", "puntos sin nombre", sin_nombre, "se cuentan igual (siguen siendo equipamientos)")
anotar("transporte.geojson", "capa", "metroplús con 1 sola estación en OSM", "", "excluido de 'sistemas_transporte'")
for inm in INMUEBLES:
    if inm["precision_coordenadas"] != "cruce":
        anotar("inmuebles_prueba.csv", inm["id_inmueble"], "coordenada aproximada (barrio)",
               inm["fuente_coordenadas"], "entorno marcado como aproximado")

bloque_entorno(ejemplo)

---
# PARTE E. Norma urbanística (POT)

## Paso 14. Bloque de norma

- Municipio fuera de la capa (Itagüí) → `no_disponible`.
- Medellín sin el archivo del POT → `pendiente_capa`.
- Con el archivo: se busca en qué polígono cae el punto (algoritmo "punto en polígono") → `encontrado` o `fuera_de_capa`.

Cuando Venture One entregue `pot_tratamientos.geojson`, basta con copiarlo en `datos/originales/geo/` y volver a correr.

In [ ]:
def punto_en_anillo(lon, lat, anillo):
    dentro, j = False, len(anillo) - 1
    for i in range(len(anillo)):
        xi, yi = anillo[i][0], anillo[i][1]
        xj, yj = anillo[j][0], anillo[j][1]
        if (yi > lat) != (yj > lat) and lon < (xj - xi) * (lat - yi) / (yj - yi) + xi:
            dentro = not dentro
        j = i
    return dentro

def punto_en_geometria(lat, lon, geometria):
    poligonos = [geometria["coordinates"]] if geometria["type"] == "Polygon" else geometria["coordinates"]
    return any(punto_en_anillo(lon, lat, exterior) and not any(punto_en_anillo(lon, lat, h) for h in huecos)
               for exterior, *huecos in poligonos)

def bloque_norma(inmueble):
    p = PARAMS["norma"]
    if inmueble["municipio"] not in p["municipios_cubiertos"]:
        return {"tratamiento_pot": None, "estado": "no_disponible",
                "motivo": f"La capa POT cubre solo: {', '.join(p['municipios_cubiertos'])}"}
    if POT is None:
        return {"tratamiento_pot": None, "estado": "pendiente_capa", "motivo": f"Falta el archivo {p['archivo_pot']}"}
    for f in POT["features"]:
        if f.get("geometry") and punto_en_geometria(inmueble["lat"], inmueble["lon"], f["geometry"]):
            return {"tratamiento_pot": f["properties"].get(p["campo_tratamiento"]), "estado": "encontrado", "motivo": ""}
    return {"tratamiento_pot": None, "estado": "fuera_de_capa", "motivo": "El punto no cae en ningún polígono"}

if POT is None:
    anotar("pot_tratamientos.geojson", "capa", "capa POT no entregada", "", "norma = pendiente_capa en Medellín")
bloque_norma(ejemplo)

---
# PARTE F. La ficha

## Paso 15. Armar la ficha JSON de cada inmueble

Juntamos todos los bloques con el formato exacto de `ejemplos/ficha_ejemplo.json` y guardamos un archivo por inmueble
en `salidas/fichas/`. También declaramos los **datos faltantes**.

In [ ]:
def datos_faltantes(inmueble, mercado):
    faltantes = []
    if not inmueble["direccion"]:
        faltantes.append("direccion")
    if inmueble["precision_coordenadas"] != "cruce":
        faltantes.append("coordenadas_exactas")
    origenes = {c["origen"] for c in mercado["comparables"]}
    comparables = [a for a in ANUNCIOS if a["origen"] in origenes]
    if not any(a["fecha"] for a in comparables):
        faltantes.append("fecha_publicacion_comparables")
    if not any(a["tipo_precio"] == "cierre" for a in comparables):
        faltantes.append("precios_de_cierre")
    return faltantes

def generar_ficha(inmueble):
    mercado = bloque_mercado(inmueble)
    return {
        "id_inmueble": inmueble["id_inmueble"],
        "inmueble": {
            "direccion": inmueble["direccion"] or None,
            "barrio": inmueble["barrio"],
            "municipio": inmueble["municipio"],
            "zona": inmueble["zona"],
            "area_m2": inmueble["area_m2"],
            "estado_texto": inmueble["estado"],
            "estado": inmueble["estado_canonico"],
            "coordenadas": {"lat": inmueble["lat"], "lon": inmueble["lon"],
                            "precision": inmueble["precision_coordenadas"],
                            "fuente": inmueble["fuente_coordenadas"]},
        },
        "mercado": mercado,
        "calidad_datos_zona": calidad_zona(inmueble["zona"]),
        "entorno": bloque_entorno(inmueble),
        "norma": bloque_norma(inmueble),
        "datos_faltantes": datos_faltantes(inmueble, mercado),
        "metadatos": {
            "version_parametros": PARAMS["version"],
            "fecha_generacion": date.today().isoformat(),
            "fecha_capas_osm": TRANSPORTE["metadata"]["fecha_osm"],
            "fuentes": ["datos/originales/anuncios_muestra.csv", "OpenStreetMap (ODbL)"]
                       + ([str(RUTA_POT)] if POT else []),
        },
    }

FICHAS = {}
for inm in INMUEBLES:
    ficha = generar_ficha(inm)
    FICHAS[inm["id_inmueble"]] = ficha
    (CARPETA_FICHAS / f"{inm['id_inmueble']}.json").write_text(
        json.dumps(ficha, ensure_ascii=False, indent=2), encoding="utf-8")
    m = ficha["mercado"]
    print(f"{inm['id_inmueble']}: {m['estado_referencia']:<13} n={m['precio_m2']['n']:<2} "
          f"mediana={m['precio_m2']['mediana']}  confianza={m['confianza']['nivel']}")

## Paso 16. Vista de lectura para la dirección

La plantilla define una "vista de una página" por inmueble. La generamos como texto (`.txt`) junto a cada JSON,
para que la dirección pueda leerla sin abrir un JSON.

In [ ]:
def pesos(valor):
    return "sin dato" if valor is None else f"${valor:,.0f}".replace(",", ".")

def vista_lectura(f):
    i, m, e, n, c = f["inmueble"], f["mercado"], f["entorno"], f["norma"], f["calidad_datos_zona"]
    p, est = m["precio_m2"], m["precio_m2_por_estado_en_zona"]
    cercana = e["estacion_mas_cercana"]
    eq = e["equipamientos_en_radio"]
    linea = "─" * 92
    return "\n".join([
        f"FICHA DE MERCADO Y ENTORNO{f['id_inmueble'] + ' · ' + f['metadatos']['fecha_generacion']:>66}",
        linea,
        f"INMUEBLE   {i['direccion'] or 'Sin dirección'} · {i['barrio']}, {i['municipio']}",
        f"           {i['area_m2']:g} m² · Estado: {i['estado']} (\"{i['estado_texto']}\")",
        f"           Ubicación: {i['coordenadas']['lat']}, {i['coordenadas']['lon']} (precisión: {i['coordenadas']['precision']})",
        "",
        f"MERCADO    Precio/m² de comparables en estado {m['estado_referencia']}",
        f"           Mediana {pesos(p['mediana'])}   Rango {pesos(p['minimo'])} – {pesos(p['maximo'])}",
        f"           Comparables: {p['n']}    Dispersión (CV): {p['coef_variacion']}",
        f"           CONFIANZA: {m['confianza']['nivel'].upper()}  →  {m['confianza']['motivo']}",
        "",
        "           Contexto de la zona (mediana por estado):",
        "           " + " · ".join(f"{k.replace('_', ' ')} {pesos(v['mediana'])} (n={v['n']})" for k, v in est.items()),
        "",
        f"ENTORNO    Radio de búsqueda: {e['radio_busqueda_m']} m   {'(aproximado)' if e['aproximado'] else ''}",
        f"           Estación más cercana: {cercana['nombre']} ({cercana['sistema']}) a {cercana['distancia_m']} m" if cercana
        else "           Estación más cercana: sin dato",
        f"           Estaciones en el radio: {e['estaciones_en_radio']}",
        f"           Equipamientos en el radio: educación {eq['educacion']} · salud {eq['salud']} · parques {eq['parques']}",
        f"                                      abastecimiento {eq['abastecimiento']} · seguridad {eq['seguridad']} · cultura {eq['cultura']}",
        "",
        f"NORMA      Tratamiento POT: {n['tratamiento_pot'] or n['estado']}  {n['motivo']}",
        "",
        f"CALIDAD    Zona: {c['anuncios_en_zona']} anuncios · {c['duplicados_eliminados']} duplicados · "
        f"{c['anuncios_validos']} válidos · inválidos: {c['anuncios_invalidos'] or 'ninguno'}",
        f"           Datos faltantes: {', '.join(f['datos_faltantes']) or 'ninguno'}",
        linea,
        f"Fuentes: {', '.join(f['metadatos']['fuentes'])} · Parámetros v{f['metadatos']['version_parametros']} "
        f"· OSM {f['metadatos']['fecha_capas_osm']}",
    ])

for id_inm, ficha in FICHAS.items():
    (CARPETA_FICHAS / f"{id_inm}.txt").write_text(vista_lectura(ficha), encoding="utf-8")

print(vista_lectura(FICHAS["INM-04"]))

---
# PARTE G. Validación contra el cálculo manual

## Paso 17. Ejecutar el validador de Venture One

`validacion/validar.py` compara nuestras fichas de INM-04, INM-06 e INM-09 con los valores calculados a mano.
Si todo coincide imprime ✓ por cada inmueble. Si algo no coincide, lista la diferencia exacta.

In [ ]:
import subprocess, sys

resultado = subprocess.run([sys.executable, "validacion/validar.py", str(CARPETA_FICHAS)],
                           capture_output=True, text=True, encoding="utf-8")
print(resultado.stdout or resultado.stderr)
print("✅ Coincide con el cálculo manual" if resultado.returncode == 0 else "❌ Revise las diferencias")

### Casos para discutir en la hackatón (vienen del cálculo manual)

1. **Duplicado ambiguo en El Salvador:** `fincaraiz:7387557` y `arrendamientos_santafe:V10746` tienen 73 m² y $300 M pero direcciones distintas.
   La regla los agrupa → INM-01 e INM-03 reciben el mismo resultado de mercado.
2. **Tres ids distintos con el mismo inmueble en Ditaires** (73 m², $490 M): ¿un apartamento publicado por varias inmobiliarias o unidades iguales?
3. **Ninguna estación a 800 m:** los inmuebles quedan a 1–2 km del Metro. El radio es un parámetro (se puede probar 1.500 m).

Estos casos son un buen argumento para la propuesta de proceso: con **matrícula inmobiliaria** como llave, la ambigüedad desaparece.

---
# PARTE H. Modelo para Power BI

## Paso 18. Construir las tablas

| Tabla | Una fila por… |
|---|---|
| `inmuebles` | inmueble (la ficha "aplanada") |
| `comparables` | par inmueble–comparable usado en su cálculo |
| `anuncios` | anuncio único, con estado canónico y motivo de invalidez |
| `puntos_entorno` | estación o equipamiento |
| `entorno_inmueble` | par inmueble–punto a ≤ 1.500 m (con marca de "dentro del radio") |
| `calidad` | problema registrado |
| `dim_zona`, `dim_estado` | zona / estado (para filtros) |

In [ ]:
NOMBRES_ZONA = {"buenos_aires_el_salvador": ("Buenos Aires – El Salvador", "Medellín"),
                "castellana_laureles": ("La Castellana – Laureles", "Medellín"),
                "ditaires_itagui": ("Ditaires", "Itagüí")}
dim_zona = pd.DataFrame([{"zona": z, "nombre_zona": n, "municipio_zona": m} for z, (n, m) in NOMBRES_ZONA.items()])

NOMBRES_ESTADO = {"remodelado": "Remodelado", "habitable": "Habitable", "por_remodelar": "Por remodelar",
                  "sin_clasificar": "Sin clasificar"}
dim_estado = pd.DataFrame([{"estado": k, "nombre_estado": v, "orden_estado": i}
                           for i, (k, v) in enumerate(NOMBRES_ESTADO.items(), start=1)])

# --- inmuebles (ficha aplanada) ---
precio_por_origen = {a["origen"]: a["precio"] for a in CRUDOS}
filas = []
for inm in INMUEBLES:
    f = FICHAS[inm["id_inmueble"]]
    i, m, e, p = f["inmueble"], f["mercado"], f["entorno"], f["mercado"]["precio_m2"]
    precio = precio_por_origen.get(inm["origen"])
    precio_m2_inm = precio / i["area_m2"] if precio else None
    fila = {
        "id_inmueble": f["id_inmueble"], "direccion": i["direccion"] or "Sin dirección", "barrio": i["barrio"],
        "municipio": i["municipio"], "zona": i["zona"], "area_m2": i["area_m2"], "estado": i["estado"],
        "estado_texto": i["estado_texto"], "lat": i["coordenadas"]["lat"], "lon": i["coordenadas"]["lon"],
        "precision_coordenadas": i["coordenadas"]["precision"], "entorno_aproximado": e["aproximado"],
        "precio_publicado": precio, "precio_m2_inmueble": round(precio_m2_inm) if precio_m2_inm else None,
        "n_comparables": p["n"], "mediana_m2": p["mediana"], "promedio_m2": p["promedio"],
        "minimo_m2": p["minimo"], "maximo_m2": p["maximo"], "coef_variacion": p["coef_variacion"],
        "confianza": m["confianza"]["nivel"], "motivo_confianza": m["confianza"]["motivo"],
        "estacion_cercana": (e["estacion_mas_cercana"] or {}).get("nombre"),
        "sistema_estacion": (e["estacion_mas_cercana"] or {}).get("sistema"),
        "distancia_estacion_m": (e["estacion_mas_cercana"] or {}).get("distancia_m"),
        "estaciones_en_radio": e["estaciones_en_radio"], "radio_m": e["radio_busqueda_m"],
        **{f"n_{k}": v for k, v in e["equipamientos_en_radio"].items()},
        "norma_estado": f["norma"]["estado"], "tratamiento_pot": f["norma"]["tratamiento_pot"],
        "datos_faltantes": ", ".join(f["datos_faltantes"]), "n_datos_faltantes": len(f["datos_faltantes"]),
    }
    # Posición frente al mercado (información, NO recomendación de compra)
    if p["mediana"] and precio_m2_inm:
        fila["diferencia_vs_mediana"] = round(precio_m2_inm / p["mediana"] - 1, 4)
    else:
        fila["diferencia_vs_mediana"] = None
    filas.append(fila)
inmuebles_bi = pd.DataFrame(filas)

# --- comparables ---
comparables_bi = pd.DataFrame([{"id_inmueble": k, **c} for k, f in FICHAS.items() for c in f["mercado"]["comparables"]])

# --- anuncios únicos ---
anuncios_bi = pd.DataFrame([{
    "origen": a["origen"], "id_anuncio": a["id_anuncio"] or "sin id", "fuente": a["fuente"], "zona": a["zona"],
    "municipio": a["municipio"], "barrio": a["barrio"] or "sin dato", "area_m2": a["area_m2"], "precio": a["precio"],
    "precio_m2": round(a["precio_m2"]) if a["precio_m2"] else None,
    "habitaciones": a["habitaciones"], "banos": a["banos"],
    "estado": a["estado_canonico"], "estado_texto": a["estado"] or "(vacío)", "tipo_precio": a["tipo_precio"],
    "es_valido": a["motivo_invalido"] is None, "motivo_invalido": a["motivo_invalido"] or "válido",
    "filas_agrupadas": len(a["origenes_agrupados"]), "origenes_agrupados": " | ".join(a["origenes_agrupados"]),
} for a in ANUNCIOS])

# --- puntos del entorno y distancias ---
puntos = []
for capa, tipo, campo in ((TRANSPORTE, "transporte", "sistema"), (EQUIPAMIENTOS, "equipamiento", "categoria")):
    for f in capa["features"]:
        lon, lat = f["geometry"]["coordinates"]
        props = f["properties"]
        puntos.append({"id_punto": props["osm_id"], "tipo": tipo, "categoria": props[campo],
                       "nombre": props["nombre"] or f"Sin nombre ({props[campo]})", "lat": lat, "lon": lon,
                       "cuenta_en_ficha": (props[campo] in PARAMS["entorno"]["sistemas_transporte"]) if tipo == "transporte"
                                          else (props[campo] in PARAMS["entorno"]["categorias_equipamientos"])})
puntos_bi = pd.DataFrame(puntos)

RADIO = PARAMS["entorno"]["radio_busqueda_m"]
DISTANCIA_MAX_M = 1500
entorno_bi = pd.DataFrame([
    {"id_inmueble": inm["id_inmueble"], "id_punto": pt["id_punto"], "distancia_m": round(d),
     "dentro_radio": d <= RADIO}
    for inm in INMUEBLES for pt in puntos
    if (d := haversine_m(inm["lat"], inm["lon"], pt["lat"], pt["lon"])) <= DISTANCIA_MAX_M
])

calidad_bi = pd.DataFrame(REGISTRO)

TABLAS = {"inmuebles": inmuebles_bi, "comparables": comparables_bi, "anuncios": anuncios_bi,
          "puntos_entorno": puntos_bi, "entorno_inmueble": entorno_bi, "calidad": calidad_bi,
          "dim_zona": dim_zona, "dim_estado": dim_estado}
for nombre, df in TABLAS.items():
    print(f"{nombre:<18} {len(df):>6} filas  {df.shape[1]:>3} columnas")

## Paso 19. Guardar el Excel para Power BI

Un solo Excel con una hoja por tabla (los decimales llegan bien sin importar la configuración regional) y además un CSV por tabla.

In [ ]:
ruta_excel = CARPETA_POWERBI / "modelo_powerbi.xlsx"
with pd.ExcelWriter(ruta_excel, engine="openpyxl") as excel:
    for nombre, df in TABLAS.items():
        df.to_excel(excel, sheet_name=nombre, index=False)
        df.to_csv(CARPETA_POWERBI / f"{nombre}.csv", sep=";", index=False, encoding="utf-8-sig")

print("✅ Guardado:", ruta_excel.resolve())
print("✅ Fichas:", sorted(p.name for p in CARPETA_FICHAS.glob("*.json")))

## Paso 20. Resumen

In [ ]:
print(f"Anuncios: {len(CRUDOS)} filas → {len(ANUNCIOS)} únicos → "
      f"{sum(a['motivo_invalido'] is None for a in ANUNCIOS)} válidos como comparables")
print(f"Fichas generadas: {len(FICHAS)} (en {CARPETA_FICHAS})")
print("Validación:", "✅ coincide" if resultado.returncode == 0 else "❌ con diferencias")
print()
inmuebles_bi[["id_inmueble", "barrio", "estado", "n_comparables", "mediana_m2", "confianza",
              "estacion_cercana", "distancia_estacion_m", "norma_estado"]]